In [1]:
#pip install spacy

In [2]:
#pip install collections

In [3]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [4]:
#!python -m spacy download en_core_web_sm

### Import required packages

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string
import math
from bs4 import BeautifulSoup
import spacy
from tqdm import trange
from collections import Counter
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from textblob import TextBlob

#### Import dataset of emails

In [6]:
df = pd.read_excel('C:/Users/dmccann/Desktop/FQ_Data.xlsx', sheet_name='Sheet1')


In [7]:
print(f"{df.shape[1]} columns and {df.shape[0]} rows")

7 columns and 6721 rows


#### Check for Null values and removal

In [8]:
df.isnull().sum()

Query Type           0
Query Subject        0
Funder               8
Business Area        0
Pobal Unit           0
Business Category    0
Summary              1
dtype: int64

In [9]:
df.dropna(inplace=True)
df.isnull().sum()

Query Type           0
Query Subject        0
Funder               0
Business Area        0
Pobal Unit           0
Business Category    0
Summary              0
dtype: int64

#### Count the number of Unique Tags for Categories

In [10]:
 df.nunique()

Query Type              7
Query Subject        6196
Funder                 10
Business Area         361
Pobal Unit             76
Business Category      12
Summary              6701
dtype: int64

#### Removal of mails with multiple Business Areas and Pobal Units by identifying delimiter '#'

In [11]:
filtered_df = df[(~df['Business Area'].str.contains('#'))]
filtered_df = filtered_df[(~filtered_df['Pobal Unit'].str.contains('#'))]
filtered_df.nunique()

Query Type              7
Query Subject        4887
Funder                 10
Business Area          53
Pobal Unit             11
Business Category      12
Summary              5232
dtype: int64

#### Cleaning of Summary Data

##### Convert Summary to lowercase

In [12]:
filtered_df['cleaned']=filtered_df['Summary'].apply(lambda x: x.lower())

##### Remove Greeting from Email Body

In [13]:
filtered_df
def remove_before_first_newline(text):
    return text.split('\n', 1)[-1]

filtered_df['cleaned'] = filtered_df['cleaned'].apply(remove_before_first_newline)


##### Remove Sign Off from Email Body

In [14]:
# Function to remove text after specified strings
def remove_after_strings(text):
    for keyword in ['\nregards', '\nkind regards', '\nmany thanks', '\nthanks']:
        if keyword in text:
            return text.split(keyword)[0]
    return text

# Apply the function to the DataFrame
filtered_df['cleaned'] = filtered_df['cleaned'].apply(remove_after_strings)

##### Clean HTML out of text string

In [15]:
# Function to remove HTML tags from a string
def clean_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

filtered_df['cleaned'] = filtered_df['cleaned'].apply(clean_html)
# Remove the string '\n' from a specific column (replace 'your_column_name' with the actual column name)
filtered_df['cleaned'] = filtered_df['cleaned'].str.replace('\n', ' ')
filtered_df

C:\Users\dmccann\AppData\Local\Temp\ipykernel_50256\2558455280.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


,Query Type,Query Subject,Funder,Business Area,Pobal Unit,Business Category,Summary,cleaned
1,FQ - Funder Query,NCS Service Calendar,DCYA,NCS,MAO,Data Analysis - Monitoring/Outputs,"Good morning, \n\nCould we please have the NCS...",could we please have the ncs calendars for 20...
2,FQ - Funder Query,PAU Confirmation,DCYA,Individual Service Profile,MAO,Data Analysis - Monitoring/Outputs,"Hi Nicole, \nHappy New Year – if we can still ...",happy new year – if we can still say that! wou...
3,EY - Internal EY Query,Extension Grant Scheme - Fee Tables,Internal EY,EYC,MAO,Data Analysis - Monitoring/Outputs,Hi All\n\nI hope you are well. \n\nThe Buildin...,i hope you are well. the building blocks ex...
4,FQ - Funder Query,Funder Queries request - NCS closure days,DCYA,NCS,MAO,Data Analysis - Monitoring/Outputs,The Department is exploring the use of closure...,information specifically required is: •\tbase...
5,FQ - Funder Query,Data request,DCYA,Equal Start,MAO,Data Analysis - Monitoring/Outputs,"Good afternoon, \n\nI would like to request th...",i would like to request the following details...
...,...,...,...,...,...,...,...,...
6716,FQ - Funder Query,Annual Budget and Staff numbers,DRCD,Corporate Reporting,FOD,Pobal,"Good morning Grace,\n\nTo follow up on my phon...",to follow up on my phone call. department of ...
6717,REP - Representation via Funder,CSP Rep,DRCD,CSP,CSSD,Application/Appraisal/Decisions,Hi \n \nI am compiling a response to the rep b...,i am compiling a response to the rep below. ...
6718,PQ - Parliamentary Question,CSP PQ,DRCD,CSP,CSSD,Application/Appraisal/Decisions,Hi Kellyann\n \nCan you please provide some ba...,can you please provide some background to in...
6719,FQ - Funder Query,TUS North East,DEASP,TUS,FOD,Data Analysis - Monitoring/Outputs,"Good afternoon,\nWill you supply me with the f...",will you supply me with the following figures ...


##### Expand Contractions

In [16]:
# Dictionary of English Contractions
contractions_dict = { "ain't": "are not","'s":" is","aren't": "are not",
                     "can't": "cannot","can't've": "cannot have",
                     "'cause": "because","could've": "could have","couldn't": "could not",
                     "couldn't've": "could not have", "didn't": "did not","doesn't": "does not",
                     "don't": "do not","hadn't": "had not","hadn't've": "had not have",
                     "hasn't": "has not","haven't": "have not","he'd": "he would",
                     "he'd've": "he would have","he'll": "he will", "he'll've": "he will have",
                     "how'd": "how did","how'd'y": "how do you","how'll": "how will",
                     "I'd": "I would", "I'd've": "I would have","I'll": "I will",
                     "I'll've": "I will have","I'm": "I am","I've": "I have", "isn't": "is not",
                     "it'd": "it would","it'd've": "it would have","it'll": "it will",
                     "it'll've": "it will have", "let's": "let us","ma'am": "madam",
                     "mayn't": "may not","might've": "might have","mightn't": "might not", 
                     "mightn't've": "might not have","must've": "must have","mustn't": "must not",
                     "mustn't've": "must not have", "needn't": "need not",
                     "needn't've": "need not have","o'clock": "of the clock","oughtn't": "ought not",
                     "oughtn't've": "ought not have","shan't": "shall not","sha'n't": "shall not",
                     "shan't've": "shall not have","she'd": "she would","she'd've": "she would have",
                     "she'll": "she will", "she'll've": "she will have","should've": "should have",
                     "shouldn't": "should not", "shouldn't've": "should not have","so've": "so have",
                     "that'd": "that would","that'd've": "that would have", "there'd": "there would",
                     "there'd've": "there would have", "they'd": "they would",
                     "they'd've": "they would have","they'll": "they will",
                     "they'll've": "they will have", "they're": "they are","they've": "they have",
                     "to've": "to have","wasn't": "was not","we'd": "we would",
                     "we'd've": "we would have","we'll": "we will","we'll've": "we will have",
                     "we're": "we are","we've": "we have", "weren't": "were not","what'll": "what will",
                     "what'll've": "what will have","what're": "what are", "what've": "what have",
                     "when've": "when have","where'd": "where did", "where've": "where have",
                     "who'll": "who will","who'll've": "who will have","who've": "who have",
                     "why've": "why have","will've": "will have","won't": "will not",
                     "won't've": "will not have", "would've": "would have","wouldn't": "would not",
                     "wouldn't've": "would not have","y'all": "you all", "y'all'd": "you all would",
                     "y'all'd've": "you all would have","y'all're": "you all are",
                     "y'all've": "you all have", "you'd": "you would","you'd've": "you would have",
                     "you'll": "you will","you'll've": "you will have", "you're": "you are",
                     "you've": "you have"}

# Regular expression for finding contractions
contractions_re=re.compile('(%s)' % '|'.join(contractions_dict.keys()))

# Function for expanding contractions
def expand_contractions(text,contractions_dict=contractions_dict):
  def replace(match):
    return contractions_dict[match.group(0)]
  return contractions_re.sub(replace, text)

# Expanding Contractions in the reviews
filtered_df['cleaned']=filtered_df['cleaned'].apply(lambda x:expand_contractions(x))

##### Remove digits and words containing digits

In [17]:
filtered_df['cleaned'] = filtered_df['cleaned'].apply(lambda x: re.sub(r'\w*\d\w*', '', x))

##### Remove Punctuations

In [18]:
filtered_df['cleaned']=filtered_df['cleaned'].apply(lambda x: re.sub('[%s]' % re.escape(string.punctuation), '', x))

In [19]:
# Removing extra spaces
filtered_df['cleaned']=filtered_df['cleaned'].apply(lambda x: re.sub(' +',' ',x))

In [20]:
# Define the function to remove bullet points from text
def remove_bullet_points(text):
    # Remove common bullet points like '-', '*', '•', etc.
    return re.sub(r'^[\s]*[-*•]\s+', '', text, flags=re.MULTILINE)

filtered_df['cleaned'] = filtered_df['cleaned'].apply(remove_bullet_points)

##### Remove Stopwords i.e. common words of language e.g. I,this,is,in etc.

In [21]:
# Loading model
nlp = spacy.load('en_core_web_sm',disable=['parser', 'ner'])

# Lemmatization with stopwords removal
filtered_df['lemmatized']=filtered_df['cleaned'].apply(lambda x: ' '.join([token.lemma_ for token in list(nlp(x)) if (token.is_stop==False)]))

##### Retry bullet point removal

In [22]:
def remove_bullet_points(s):
    # Define the bullet point pattern
    pattern = re.compile(r'•\s*')
    # Remove bullet points from the string
    return pattern.sub('', s)

filtered_df['lemmatized'] = filtered_df['lemmatized'].apply(remove_bullet_points)

#### remove non-alphabetic characters

In [23]:
def remove_non_alphabetic(s):
    # Remove non-alphabetic characters using regex
    return re.sub(r'[^a-zA-Z\s]', '', s)

filtered_df['lemmatized'] = filtered_df['lemmatized'].apply(remove_non_alphabetic)

#### Remove any greetings or sign offs

In [24]:
def remove_specific_words(s):
    # Define the words to remove
    words_to_remove = ['hi', 'hello', 'thank', 'regard', 'kind']
    # Create a regex pattern to match any of the words
    pattern = re.compile(r'\b(?:' + '|'.join(words_to_remove) + r')\b', re.IGNORECASE)
    # Remove the words from the string
    return pattern.sub('', s)

filtered_df['lemmatized']= filtered_df['lemmatized'].apply(remove_specific_words)


#### Remove empty rows or whitespace rows

In [25]:
filtered_df = filtered_df[filtered_df['lemmatized'].apply(lambda x: not pd.isna(x) and x.strip() != '')]

In [26]:
filtered_df

,Query Type,Query Subject,Funder,Business Area,Pobal Unit,Business Category,Summary,cleaned,lemmatized
1,FQ - Funder Query,NCS Service Calendar,DCYA,NCS,MAO,Data Analysis - Monitoring/Outputs,"Good morning, \n\nCould we please have the NCS...",could we please have the ncs calendars for an...,ncs calendar service
2,FQ - Funder Query,PAU Confirmation,DCYA,Individual Service Profile,MAO,Data Analysis - Monitoring/Outputs,"Hi Nicole, \nHappy New Year – if we can still ...",happy new year – if we can still say that woul...,happy new year able confirm pau email address...
3,EY - Internal EY Query,Extension Grant Scheme - Fee Tables,Internal EY,EYC,MAO,Data Analysis - Monitoring/Outputs,Hi All\n\nI hope you are well. \n\nThe Buildin...,i hope you are well the building blocks exten...,hope building block extension grant applicat...
4,FQ - Funder Query,Funder Queries request - NCS closure days,DCYA,NCS,MAO,Data Analysis - Monitoring/Outputs,The Department is exploring the use of closure...,information specifically required is •\tbased...,information specifically require base ncs pr...
5,FQ - Funder Query,Data request,DCYA,Equal Start,MAO,Data Analysis - Monitoring/Outputs,"Good afternoon, \n\nI would like to request th...",i would like to request the following details...,like request following detail equal start se...
...,...,...,...,...,...,...,...,...,...
6716,FQ - Funder Query,Annual Budget and Staff numbers,DRCD,Corporate Reporting,FOD,Pobal,"Good morning Grace,\n\nTo follow up on my phon...",to follow up on my phone call department of r...,follow phone department rural community deve...
6717,REP - Representation via Funder,CSP Rep,DRCD,CSP,CSSD,Application/Appraisal/Decisions,Hi \n \nI am compiling a response to the rep b...,i am compiling a response to the rep below we...,compile response rep material previously sub...
6718,PQ - Parliamentary Question,CSP PQ,DRCD,CSP,CSSD,Application/Appraisal/Decisions,Hi Kellyann\n \nCan you please provide some ba...,can you please provide some background to inf...,provide background inform response pq
6719,FQ - Funder Query,TUS North East,DEASP,TUS,FOD,Data Analysis - Monitoring/Outputs,"Good afternoon,\nWill you supply me with the f...",will you supply me with the following figures ...,supply follow figure number participant curren...


In [27]:
filtered_df.to_excel('test.xlsx',index=False)

PermissionError: [Errno 13] Permission denied: 'test.xlsx'

In [ ]:
def corpus(text):
    text_list = text.split()
    return text_list

filtered_df['Summary_list'] = filtered_df['lemmatized'].apply(corpus)

In [ ]:
filtered_df = filtered_df.reset_index(drop=True)

In [ ]:
corpus = []
for i in trange(filtered_df.shape[0], ncols=150, nrows=10, colour='green', smoothing=0.8):
    corpus.extend(filtered_df['Summary_list'][i])

print(len(corpus))

In [ ]:
mostCommon = Counter(corpus).most_common(10)
mostCommon

In [ ]:
words = []
freq = []
for word,count in mostCommon:
    words.append(word)
    freq.append(count)

In [ ]:
sns.barplot(x=freq, y=words,hue=words)
plt.title('Top ten most frequently Occuring Words')
plt.show()

In [ ]:
cv =  CountVectorizer(ngram_range=(2,2))
bigrams = cv.fit_transform(filtered_df['lemmatized'])

In [ ]:
count_values = bigrams.toarray().sum(axis=0)
ngram_freq = pd.DataFrame(sorted([(count_values[i], k) for k, i in cv.vocabulary_.items()], reverse = True))
ngram_freq.columns = ['frequency','ngram']

In [ ]:
sns.barplot(x=ngram_freq['frequency'][:10], y = ngram_freq['ngram'][:10], hue=ngram_freq['ngram'][:10])
plt.title('Top 10 Most Frequently Occurring Bigrams')
plt.show()

In [ ]:
cv =  CountVectorizer(ngram_range=(3,3))
trigrams = cv.fit_transform(filtered_df['lemmatized'])

In [ ]:
count_values = trigrams.toarray().sum(axis=0)
ngram_freq = pd.DataFrame(sorted([(count_values[i], k) for k, i in cv.vocabulary_.items()], reverse = True))
ngram_freq.columns = ['frequency','ngram']

In [ ]:
sns.barplot(x=ngram_freq['frequency'][:10], y = ngram_freq['ngram'][:10], hue=ngram_freq['ngram'][:10])
plt.title('Top 10 Most Frequently Occurring Trigrams')
plt.show()

In [ ]:
#ngram_freq.to_excel('trigrams.xlsx',index=False)

In [ ]:
blob  =  TextBlob(str(filtered_df['Summary']))
pos_df = pd.DataFrame(blob.tags,columns=['word','pos'])


# Dictionary mapping POS tags to descriptions
pos_descriptions = {
    'CC': 'Coordinating conjunction',
    'CD': 'Cardinal number',
    'DT': 'Determiner',
    'EX': 'Existential there',
    'FW': 'Foreign word',
    'IN': 'Preposition or subordinating conjunction',
    'JJ': 'Adjective',
    'JJR': 'Adjective, comparative',
    'JJS': 'Adjective, superlative',
    'LS': 'List item marker',
    'MD': 'Modal',
    'NN': 'Noun, singular or mass',
    'NNS': 'Noun, plural',
    'NNP': 'Proper noun, singular',
    'NNPS': 'Proper noun, plural',
    'PDT': 'Predeterminer',
    'POS': 'Possessive ending',
    'PRP': 'Personal pronoun',
    'PRP$': 'Possessive pronoun',
    'RB': 'Adverb',
    'RBR': 'Adverb, comparative',
    'RBS': 'Adverb, superlative',
    'RP': 'Particle',
    'SYM': 'Symbol',
    'TO': 'to',
    'UH': 'Interjection',
    'VB': 'Verb, base form',
    'VBD': 'Verb, past tense',
    'VBG': 'Verb, gerund or present participle',
    'VBN': 'Verb, past participle',
    'VBP': 'Verb, non-3rd person singular present',
    'VBZ': 'Verb, 3rd person singular present',
    'WDT': 'Wh-determiner',
    'WP': 'Wh-pronoun',
    'WP$': 'Possessive wh-pronoun',
    'WRB': 'Wh-adverb'
}

# Replace POS tags with descriptions
pos_df['pos'] = pos_df['pos'].map(pos_descriptions)

top_pos = pos_df['pos'].value_counts()

In [ ]:
top_pos

In [ ]:
plt.figure(figsize=(8,10))
sns.barplot(y=top_pos.index, x=top_pos.values,hue=top_pos.index)
plt.title('Part of Speech tagging in Summary')
plt.show()